# Surgical Instrument Force & Motion Analysis — 2 · Visualization

**This is the second of two notebooks.** It reads the single JSON file written by
**`surgical_force_processing.ipynb`** and reproduces the full analysis — every figure
and table — without touching the raw `.igs.mha` data or recomputing any metric.

```
surgical_force_processing.ipynb   →   analysis_data.json   →   surgical_force_visualization.ipynb
        (read + compute)                 (single file)              (plot everything)
```

Run the processing notebook first so `DATA_JSON` exists, then run this notebook top to
bottom.


## Setup

In [ ]:
# Visualization needs only numpy + matplotlib (metrics are precomputed).
import importlib, subprocess, sys
for pkg in ("numpy", "matplotlib", "pandas"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, warnings
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# Masked (not-in-use) frames are NaN; nanmean/nanmax over all-NaN slices is expected.
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

# ---- Plot style ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})
print("Setup complete.")


## Load the metrics JSON

Read `DATA_JSON` (written by the processing notebook) and rebuild the in-memory
structures the plotting cells expect. Constants (instrument list, pairs, thresholds)
travel in the file's `meta`; the colors / colormap / units are visualization choices
defined here.


In [ ]:
# ---- Load the metrics JSON produced by the processing notebook ------------------
DATA_JSON = "data/analysis_data.json"   # written by surgical_force_processing.ipynb

with open(DATA_JSON) as fh:
    payload = json.load(fh)             # NaN tokens are read back as float('nan')

meta = payload["meta"]
INSTRUMENTS   = meta["instruments"]
PAIRS         = [tuple(p) for p in meta["pairs"]]
PAIR_KEYS     = meta["pair_keys"]
SMOOTH_WINDOW = meta["smooth_window"]
IDLE_SPEED    = meta["idle_speed"]
VOXEL_SIZE    = meta["voxel_size"]
PLANE_DIST_MAX = meta["plane_dist_max"]
TIME_UNIT     = meta["time_unit"]

# ---- Visualization constants (colors / colormap / units) ------------------------
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
TIME_CMAP = "viridis"
PAIR_COLOR = {"Bipolar-Cavitron": "#7A5195", "Bipolar-Scissors": "#EF5675"}
KIN_UNITS = {"velocity": "mm/s", "acceleration": "mm/s²", "jerk": "mm/s³"}

def trial_colors(n):
    "light->dark ramp to distinguish the trials within one participant"
    return plt.cm.cividis(np.linspace(0.15, 0.85, max(n, 1)))

# ---- Rebuild the in-memory trial structures the plots expect --------------------
def _np(x):
    return np.asarray(x, float)

trials = []
participants = {pid: [] for pid in meta["participants"]}
for tj in payload["trials"]:
    insts = tj["present_instruments"]
    tr = {
        "participant": tj["participant"],
        "trial": tj["trial"],
        "name": tj["name"],
        "label": tj["label"],
        "duration": tj["duration"],
        "rmse": tj["rmse"],
        # membership dict used by the per-instrument summaries (values unused)
        "pos": {inst: None for inst in insts},
        "tnorm": _np(tj["tnorm"]),
        "fmag": _np(tj["fmag"]),
        "dFdt": _np(tj["dFdt"]),
        "kin": {inst: {m: _np(tj["kin"][inst][m]) for m in tj["kin"][inst]} for inst in tj["kin"]},
        "pos_plot": {inst: _np(v) for inst, v in tj["pos_plot"].items()},
        "ang_speed": {inst: _np(v) for inst, v in tj["ang_speed"].items()},
        "ang_accel": {inst: _np(v) for inst, v in tj["ang_accel"].items()},
        "tracking_status": {inst: np.asarray(v, int) for inst, v in tj["tracking_status"].items()},
        "dist": {pair: _np(v) for pair, v in tj["dist"].items()},
        "angle": {pair: _np(v) for pair, v in tj["angle"].items()},
        "force_cov": tj["force_cov"],
        "dFdt_abs_mean": tj["dFdt_abs_mean"],
        "inuse_frac": tj["inuse_frac"],
        "pathlen": tj["pathlen"],
        "netdisp": tj["netdisp"],
        "straightness": tj["straightness"],
        "bbox_vol": tj["bbox_vol"],
        "idle_frac": tj["idle_frac"],
        "ell_vol": tj["ell_vol"],
        "aniso": tj["aniso"],
        "occ_eff": tj["occ_eff"],
    }
    trials.append(tr)
    participants[tr["participant"]].append(tr)

print(f"loaded {len(trials)} trial(s) across {len(participants)} participant(s) from {DATA_JSON}")


## 6 · Force magnitude and rate of force change — per participant

Two per-participant figures over normalized trial time — **|force|** `√(fx²+fy²+fz²)` and its **rate of change** **|dF/dt|** (N/s). Each line is one trial; a steadier hand keeps `|dF/dt|` low and flat across the trial.

In [ ]:
n = len(participants)
ncol = min(3, n); nrow = int(np.ceil(n / ncol))

def _per_participant_timeseries(signal_key, ylabel, suptitle):
    "One panel per participant; one line per trial over normalized time."
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.2 * nrow),
                             squeeze=False, sharex=True)
    for ax, (pid, ptrials) in zip(axes.flat, participants.items()):
        cols = trial_colors(len(ptrials))
        for tr, c in zip(ptrials, cols):
            ax.plot(tr["tnorm"], tr[signal_key], lw=1.0, color=c, label=f"T{tr['trial']}")
        ax.set_title(f"Participant {pid}")
        ax.set_xlabel("normalized time"); ax.set_ylabel(ylabel)
        ax.margins(x=0); ax.legend(title="trial", fontsize=8)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    fig.suptitle(suptitle, fontweight="bold")
    fig.tight_layout()
    return fig

_per_participant_timeseries("fmag", "|force|  (N)",
                            "Force magnitude  √(fx²+fy²+fz²)  per participant")
_per_participant_timeseries("dFdt", "|dF/dt|  (N/s)",
                            "Rate of force change  |dF/dt|  per participant")
plt.show()


## 7 · Velocity, acceleration and jerk — one figure per participant

Rows are the three motion metrics; columns are the instruments. Each line is one of
the participant's trials, against normalized time.

In [ ]:
metrics = ["velocity", "acceleration", "jerk"]
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(len(metrics), len(INSTRUMENTS),
                             figsize=(4.6 * len(INSTRUMENTS), 2.9 * len(metrics)),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for r, metric in enumerate(metrics):
        for c, inst in enumerate(INSTRUMENTS):
            ax = axes[r][c]
            for tr, col in zip(ptrials, cols):
                if inst in tr["kin"]:
                    ax.plot(tr["tnorm"], tr["kin"][inst][metric], lw=0.9, color=col,
                            label=f"T{tr['trial']}" if (r == 0 and c == 0) else None)
            if r == 0:
                ax.set_title(inst, color=INST_COLOR[inst])
            if c == 0:
                ax.set_ylabel(f"{metric}\n({KIN_UNITS[metric]})")
            if r == len(metrics) - 1:
                ax.set_xlabel("normalized time")
            ax.margins(x=0)
    axes[0][0].legend(title="trial", fontsize=8, loc="upper right")
    fig.suptitle(f"Motion derivatives — participant {pid}", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 8 · 3D instrument trajectories — one figure per participant

Rows are the participant's trials, columns are the instruments. Each registered tip
path is drawn as a 3D line colored from the start (dark) to the end (yellow) of the
recording. Only **in-use** frames are drawn (§5) — the line breaks over frames where
the instrument is untracked or its tip leaves the working area (>50 mm from the registration plane, or outside the fiducial polygon). All
trials share the common registered frame. Within each instrument column every panel uses the **same square (equal-aspect) axis cube** — shared x/y/z limits fixed from that instrument's paths across all participants — so e.g. all Bipolar panels are at one scale and directly comparable across participants and trials.


In [ ]:
def _color_line3d(ax, xyz, tnorm, lw=1.6):
    pts = xyz.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    good = ~np.isnan(segs).any(axis=(1, 2))          # skip segments touching a masked frame
    lc = Line3DCollection(segs[good], cmap=TIME_CMAP, array=tnorm[:-1][good], linewidth=lw)
    lc.set_clim(0, 1)                                # fix color scale to normalized time [0,1]
    ax.add_collection3d(lc)
    return lc


# ---- Per-instrument shared axis limits so panels are comparable across participants ---
# All trials live in the common registered frame (§4). For each instrument we fix one
# equal-aspect cube (a square/cubic box: identical span on x, y and z) from the combined
# extent of that instrument's in-use paths across every participant and trial. Every
# panel for that instrument then reuses it, so e.g. all Bipolar paths render at the same
# scale and can be compared across participants; different instruments may use different
# cubes so each is sized to its own working volume.
def _cube_lims(pos_list):
    "Equal-aspect (cubic) x/y/z limits covering all trajectories in pos_list, or None."
    valid = [P for P in pos_list if P is not None and np.any(~np.isnan(P))]
    if not valid:
        return None
    stack = np.concatenate(valid, axis=0)
    lo, hi = np.nanmin(stack, axis=0), np.nanmax(stack, axis=0)
    mid = (lo + hi) / 2
    half = (np.nanmax(hi - lo) / 2) or 1             # half-edge of the common cube
    return [(m - half, m + half) for m in mid]       # (x, y, z), equal span -> square aspect

INST_AXIS_LIMS = {inst: _cube_lims([tr["pos_plot"].get(inst) for tr in trials])
                  for inst in INSTRUMENTS}

for pid, ptrials in participants.items():
    nrow, ncol = len(ptrials), len(INSTRUMENTS)
    fig = plt.figure(figsize=(5.6 * ncol, 4.6 * nrow))
    lc = None
    for r, tr in enumerate(ptrials):
        for c, inst in enumerate(INSTRUMENTS):
            ax = fig.add_subplot(nrow, ncol, r * ncol + c + 1, projection="3d")
            P = tr["pos_plot"].get(inst)                 # in-use frames only (rest are NaN)
            if P is None or not np.any(~np.isnan(P)):
                ax.set_axis_off(); continue
            lc = _color_line3d(ax, P, tr["tnorm"])
            ax.set_title(f"T{tr['trial']} · {inst}", color=INST_COLOR[inst], fontsize=10)
            ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
            # Shared per-instrument cube when available, else a per-panel cube.
            lims = INST_AXIS_LIMS.get(inst)
            if lims is None:
                rng = (np.nanmax(np.nanmax(P, 0) - np.nanmin(P, 0)) / 2) or 1
                lims = [(m - rng, m + rng) for m in np.nanmean(P, axis=0)]
            for setlim, (lo, hi) in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), lims):
                setlim(lo, hi)
            ax.set_box_aspect((1, 1, 1))                 # render the cube as a true square box
            ax.view_init(elev=20, azim=-60)
    if lc is not None:
        cb = fig.colorbar(lc, ax=fig.axes, shrink=0.5, pad=0.02)
        cb.set_label("normalized time")
    fig.suptitle(f"Registered tip trajectories — participant {pid}", fontweight="bold")
    plt.show()

## 9 · Inter-instrument distance

Tip-to-tip distance over time (**along-trial**, one figure per participant) and its
trial average (**summative**, grouped by participant), for the pairs used together:
**Bipolar–Cavitron** and **Bipolar–Scissors**. Frames where either instrument is not in
use (§5 working-area mask) are set to NaN — the line breaks there and those frames are
excluded from the averages. Small distances mean the
instruments are working close together (bimanual coordination / proximity analysis).

In [ ]:
present_pairs = [k for k in PAIR_KEYS if any(k in tr["dist"] for tr in trials)]

# --- along-trial: one figure per participant, one subplot per pair ---
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(present_pairs),
                             figsize=(5.2 * len(present_pairs), 3.4),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, pair in enumerate(present_pairs):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if pair in tr["dist"]:
                ax.plot(tr["tnorm"], tr["dist"][pair], lw=1.0, color=col, label=f"T{tr['trial']}")
        ax.set_title(pair.replace("-", " – "), color=PAIR_COLOR.get(pair, "#333"))
        ax.set_xlabel("normalized time"); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("distance (mm)"); ax.legend(title="trial", fontsize=8)
    fig.suptitle(f"Inter-instrument distance — participant {pid}", fontweight="bold")
    fig.tight_layout(); plt.show()

# --- summative: mean distance per pair, grouped by participant (NaN-aware) ---
def _nanmean(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

def _pair_grouped_bar(value_of, ylabel, title):
    pids = list(participants)
    fig, ax = plt.subplots(figsize=(1.8 * len(pids) + 4, 4.2))
    x = np.arange(len(pids)); k = max(len(present_pairs), 1); w = 0.8 / k
    for i, pair in enumerate(present_pairs):
        means, errs = [], []
        for pid in pids:
            vals = [value_of(tr, pair) for tr in participants[pid] if pair in tr["dist"]]
            vals = [v for v in vals if v is not None and np.isfinite(v)]
            means.append(np.mean(vals) if vals else 0.0)
            errs.append(np.std(vals) if len(vals) > 1 else 0.0)
        means, errs = np.asarray(means), np.asarray(errs)
        ax.bar(x + (i - (k - 1) / 2) * w, means, width=w * 0.95,
               yerr=np.vstack([np.minimum(errs, means), errs]),
               label=pair.replace("-", " – "), color=PAIR_COLOR.get(pair, None),
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x); ax.set_xticklabels(pids); ax.set_xlabel("participant")
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8); ax.margins(y=0.15)
    fig.tight_layout(); plt.show()

_pair_grouped_bar(lambda tr, pair: _nanmean(tr["dist"][pair]),
                  "mean distance (mm)", "Average inter-instrument distance (in-use frames)")

## 10 · Instrument orientation

Angle between the two instruments' **long axes** over time (**along-trial**) and its
trial average (**summative**), for **Bipolar–Cavitron** and **Bipolar–Scissors**. The
long axis is the pivot (shaft) direction recovered in §5; 0° means the two shafts are
parallel, 180° anti-parallel. Frames where either instrument is untracked are NaN.

In [ ]:
present_ang = [k for k in PAIR_KEYS if any(k in tr["angle"] for tr in trials)]

# --- along-trial angle: one figure per participant, one subplot per pair ---
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(present_ang),
                             figsize=(5.2 * len(present_ang), 3.4),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, pair in enumerate(present_ang):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if pair in tr["angle"]:
                ax.plot(tr["tnorm"], tr["angle"][pair], lw=1.0, color=col, label=f"T{tr['trial']}")
        ax.set_title(pair.replace("-", " – "), color=PAIR_COLOR.get(pair, "#333"))
        ax.set_xlabel("normalized time"); ax.set_ylim(0, 180); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("angle (deg)"); ax.legend(title="trial", fontsize=8)
    fig.suptitle(f"Inter-instrument angle — participant {pid}", fontweight="bold")
    fig.tight_layout(); plt.show()

# --- summative: mean angle per pair, grouped by participant ---
def _nanmean_a(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

pids = list(participants)
fig, ax = plt.subplots(figsize=(1.8 * len(pids) + 4, 4.2))
x = np.arange(len(pids)); k = max(len(present_ang), 1); w = 0.8 / k
for i, pair in enumerate(present_ang):
    means, errs = [], []
    for pid in pids:
        vals = [_nanmean_a(tr["angle"][pair]) for tr in participants[pid] if pair in tr["angle"]]
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        means.append(np.mean(vals) if vals else 0.0)
        errs.append(np.std(vals) if len(vals) > 1 else 0.0)
    means, errs = np.asarray(means), np.asarray(errs)
    ax.bar(x + (i - (k - 1) / 2) * w, means, width=w * 0.95,
           yerr=np.vstack([np.minimum(errs, means), errs]),
           label=pair.replace("-", " – "), color=PAIR_COLOR.get(pair, None),
           error_kw=dict(lw=1, capsize=3, ecolor="#555"))
ax.set_xticks(x); ax.set_xticklabels(pids); ax.set_xlabel("participant")
ax.set_ylabel("mean angle (deg)"); ax.set_title("Average inter-instrument angle")
ax.legend(fontsize=8); ax.margins(y=0.15)
fig.tight_layout(); plt.show()

## 10a · Instrument angular speed

Rotational speed of each instrument's frame over the procedure, plotted against
**normalized trial time** — one figure per participant, one column per instrument, one
line per trial. Angular speed is the per-frame rotation magnitude of the tip pose divided
by the frame interval (deg/s), recovered in §5. Frames where the instrument is not in use
(untracked, or gated out by the working-area rule: >50 mm from the registration plane, or outside the fiducial polygon) are NaN and left out. Values above
the 95th percentile (pooled over all trials) are dropped as outliers, so the shared scale
isn't dominated by brief spikes.

In [ ]:
# --- angular speed along normalized time: one figure per participant, one col per instrument ---
# drop outliers above the 95th percentile, then share one y-scale across all
# participants & instruments so the figures are directly comparable
_sp_all = np.concatenate([np.asarray(tr["ang_speed"][inst], float).ravel()
                          for tr in trials for inst in tr.get("ang_speed", {})]) \
          if trials else np.array([np.nan])
sp_p95 = float(np.nanpercentile(_sp_all, 95)) if np.any(np.isfinite(_sp_all)) else 1.0
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(INSTRUMENTS),
                             figsize=(4.6 * len(INSTRUMENTS), 3.0),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, inst in enumerate(INSTRUMENTS):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if inst in tr.get("ang_speed", {}):
                y = np.where(tr["ang_speed"][inst] <= sp_p95, tr["ang_speed"][inst], np.nan)
                ax.plot(tr["tnorm"], y, lw=0.9, color=col,
                        label=f"T{tr['trial']}" if c == 0 else None)
        ax.set_title(inst, color=INST_COLOR[inst])
        ax.set_xlabel("normalized time"); ax.set_ylim(0, sp_p95 * 1.05); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("angular speed (deg/s)")
    axes[0][0].legend(title="trial", fontsize=8, loc="upper right")
    fig.suptitle(f"Instrument angular speed — participant {pid}", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 10b · Instrument angular acceleration

Rate of change of the angular speed above, plotted against **normalized trial time** —
same layout (one figure per participant, one column per instrument, one line per trial).
Computed in §5 as the time-derivative of the (lightly smoothed) angular speed, so units
are deg/s²; positive means the instrument's rotation is speeding up, negative slowing
down. Non-in-use frames are NaN, and values whose magnitude exceeds the 95th percentile
(pooled over all trials) are dropped as outliers.

In [ ]:
# --- angular acceleration along normalized time: one figure per participant, one col per instrument ---
# drop outliers whose magnitude exceeds the 95th percentile, then share one
# zero-symmetric y-scale across all participants & instruments
_ac_all = np.concatenate([np.asarray(tr["ang_accel"][inst], float).ravel()
                          for tr in trials for inst in tr.get("ang_accel", {})]) \
          if trials else np.array([np.nan])
ac_p95 = float(np.nanpercentile(np.abs(_ac_all), 95)) if np.any(np.isfinite(_ac_all)) else 1.0
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(INSTRUMENTS),
                             figsize=(4.6 * len(INSTRUMENTS), 3.0),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, inst in enumerate(INSTRUMENTS):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if inst in tr.get("ang_accel", {}):
                y = np.where(np.abs(tr["ang_accel"][inst]) <= ac_p95, tr["ang_accel"][inst], np.nan)
                ax.plot(tr["tnorm"], y, lw=0.9, color=col,
                        label=f"T{tr['trial']}" if c == 0 else None)
        ax.axhline(0, color="#999", lw=0.6, zorder=0)
        ax.set_title(inst, color=INST_COLOR[inst])
        ax.set_xlabel("normalized time"); ax.set_ylim(-ac_p95 * 1.05, ac_p95 * 1.05); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("angular acceleration (deg/s²)")
    axes[0][0].legend(title="trial", fontsize=8, loc="upper right")
    fig.suptitle(f"Instrument angular acceleration — participant {pid}", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 11 · Summative figures

Cross-participant comparison as grouped bars (x = participant). **Force** is
summarized per participant (single sensor); **velocity / acceleration / jerk / path
length / straightness / % tracked / angular speed** are per instrument. **% tracked**
is the share of frames with valid (non-frozen) tracking. Error bars are the standard
deviation across each participant's trials (clipped at zero, since the quantities are
non-negative).

In [ ]:
def _grouped_bar(ax, participant_ids, series, ylabel, title, colors=None):
    "series: dict label -> (means[np], errs[np]); one group of bars per participant."
    x = np.arange(len(participant_ids))
    k = len(series)
    w = 0.8 / k
    for i, (lab, (means, errs)) in enumerate(series.items()):
        means, errs = np.asarray(means, float), np.asarray(errs, float)
        yerr = np.vstack([np.minimum(errs, means), errs])
        off = (i - (k - 1) / 2) * w
        ax.bar(x + off, means, width=w * 0.95, yerr=yerr, label=lab,
               color=None if colors is None else colors[i],
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x); ax.set_xticklabels(participant_ids)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_xlabel("participant")
    ax.margins(y=0.15)
    if k > 1:
        ax.legend(fontsize=8)

pids = list(participants)

# mean/std of any per-trial scalar, per instrument, across each participant's trials
def inst_group_stats(value_fn):
    means, errs = {}, {}
    for inst in INSTRUMENTS:
        m, e = [], []
        for pid in pids:
            vals = [value_fn(tr, inst) for tr in participants[pid] if inst in tr["pos"]]
            vals = [v for v in vals if v is not None and np.isfinite(v)]
            m.append(np.mean(vals) if vals else 0.0)
            e.append(np.std(vals) if len(vals) > 1 else 0.0)
        means[inst], errs[inst] = m, e
    return means, errs

fig, axes = plt.subplots(4, 2, figsize=(13, 17))

# (a) average force magnitude per participant (single sensor)
fmeans = [np.mean([tr["fmag"].mean() for tr in participants[pid]]) for pid in pids]
ferrs = [np.std([tr["fmag"].mean() for tr in participants[pid]]) if len(participants[pid]) > 1
         else 0.0 for pid in pids]
_grouped_bar(axes[0][0], pids, {"|force|": (fmeans, ferrs)},
             "|force|  (N)", "Average force magnitude per participant", colors=["#0072B2"])

# (b–g) per-instrument metrics
panels = [
    (axes[0][1], "velocity",     KIN_UNITS["velocity"],     lambda tr, i: np.nanmean(tr["kin"][i]["velocity"])),
    (axes[1][0], "acceleration", KIN_UNITS["acceleration"], lambda tr, i: np.nanmean(tr["kin"][i]["acceleration"])),
    (axes[1][1], "jerk",         KIN_UNITS["jerk"],         lambda tr, i: np.nanmean(tr["kin"][i]["jerk"])),
    (axes[2][0], "path length",  "mm",                      lambda tr, i: tr["pathlen"][i]),
    (axes[2][1], "straightness", "net / path",              lambda tr, i: tr["straightness"][i]),
    (axes[3][0], "% tracked",    "% of frames",             lambda tr, i: 100 * tr["tracking_status"][i].mean()),
    (axes[3][1], "angular speed", "deg/s",                  lambda tr, i: np.nanmean(tr["ang_speed"][i])),
]
for ax, name, unit, fn in panels:
    means, errs = inst_group_stats(fn)
    series = {inst: (means[inst], errs[inst]) for inst in INSTRUMENTS}
    _grouped_bar(ax, pids, series, unit, f"Average {name} per instrument",
                 colors=[INST_COLOR[i] for i in INSTRUMENTS])
axes[3][0].set_ylim(0, 100)                       # % tracked is a percentage

fig.suptitle("Cross-participant comparison", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## 11a · Force-control metrics across participants

The two force-control indicators are single-sensor per-trial scalars (like force
magnitude), so each is shown as **one bar per participant** with error bars = the
**standard deviation across that participant's trials**:

- **Coefficient of variation** `CoV = SD(|F|) / mean(|F|) × 100%` — normalised force
  variability; **lower = steadier** force control.
- **Mean absolute rate of force change** `⟨|dF/dt|⟩` (N/s) — **lower = more controlled,
  gradual** force application.


In [ ]:
# Force-control metrics are single-sensor per trial (like |force|): one bar per
# participant, error bars = SD across that participant's trials.
def _force_stats(value_fn):
    means, errs = [], []
    for pid in pids:
        vals = [value_fn(tr) for tr in participants[pid]]
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        means.append(np.mean(vals) if vals else 0.0)
        errs.append(np.std(vals) if len(vals) > 1 else 0.0)
    return means, errs

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cov_m, cov_e = _force_stats(lambda tr: tr["force_cov"])
_grouped_bar(axes[0], pids, {"CoV": (cov_m, cov_e)},
             "CoV  (%)", "Force coefficient of variation per participant",
             colors=["#0072B2"])
dfdt_m, dfdt_e = _force_stats(lambda tr: tr["dFdt_abs_mean"])
_grouped_bar(axes[1], pids, {"|dF/dt|": (dfdt_m, dfdt_e)},
             "|dF/dt|  (N/s)", "Mean absolute rate of force change per participant",
             colors=["#D55E00"])
fig.suptitle("Force-control metrics across participants", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()


## 11b · Instrument-use localization (spread & concentration)

Orientation-invariant complements to the axis-aligned working volume: the **covariance-ellipsoid volume** `(4/3)π·√(λ₁λ₂λ₃)` measures overall spread regardless of how the cloud sits in the frame; its **anisotropy** `λ₁/λ₃` says whether motion is confined to a line/plane (large) or an isotropic blob (≈1); and the dwell-time-weighted **occupancy `exp(H)`** is the effective number of occupied 2 mm voxels. Because the number of voxels a tip touches keeps growing the longer a trial lasts, it is **normalized by trial duration** (`exp(H) / T`, eff. voxels/s) so participants are comparable regardless of how long they took. Smaller ellipsoid volume / occupancy rate ⇒ more localized use.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
loc_panels = [
    (axes[0], "ellipsoid volume", "mm\u00b3",       lambda tr, i: tr["ell_vol"][i]),
    (axes[1], "anisotropy \u03bb\u2081/\u03bb\u2083", "ratio",      lambda tr, i: tr["aniso"][i]),
    # occupancy grows with how long a trial lasts, so divide by trial duration:
    # eff. voxels touched per second, making participants comparable regardless of
    # how much time they spent on the task.
    (axes[2], "occupancy exp(H) / time", "eff. voxels/s",
     lambda tr, i: tr["occ_eff"][i] / tr["duration"]),
]
for ax, name, unit, fn in loc_panels:
    means, errs = inst_group_stats(fn)
    series = {inst: (means[inst], errs[inst]) for inst in INSTRUMENTS}
    _grouped_bar(ax, pids, series, unit, name, colors=[INST_COLOR[i] for i in INSTRUMENTS])
fig.suptitle("Instrument-use localization per participant", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## 12 · Force pyramid — spatial distribution of applied force

The **force pyramid** (Sawaya et al., *Operative Neurosurgery* 2017) maps *where* force is
applied across the registered workspace, rather than *how much* over time. Following the
paper's *Spatial Analysis*:

1. each in-use tip's registered `x y z` is rounded to the nearest **0.5 mm** grid;
2. the forces of all samples sharing an `xyz` voxel are **averaged**;
3. those per-voxel forces are **summed along the z-axis** (depth) to give the total force
   at each `xy` cell — the pyramid.

Every recorded frame is used **as-is** — no resampling to a fixed rate; whatever timestamps
the recording contains are what feed the grid.

**Handling co-active instruments.** This dataset records a **single overall `|force|` per
timestamp** (not one force per instrument), so when more than one instrument is in the active
space at the same frame a rule is needed to decide where that one reading lands. Two are
provided via `FORCE_ATTRIBUTION`:

- **`"split"`** (default) — the reading is shared **equally** among the instruments in use at
  that frame (`|force| / n_active`), so a co-active frame contributes exactly the sensor
  reading to the map (no double-counting). Pooled across all in-use tips → a *bimanual/total*
  pyramid.
- **`"primary"`** — the full `|force|` is attributed to a single **primary** instrument
  (`PRIMARY_INSTRUMENT`, e.g. `"Cavitron"` the aspirator or `"Scissors"`), and the pyramid is
  built from that instrument's in-use frames only — closest to the paper's per-hand pyramid.

Only in-use frames contribute either way: untracked / out-of-working-area frames are `NaN` in
`pos_plot` (§5) and dropped, exactly as the in-use masking used elsewhere.

The pyramid is built **per trial**: one figure per participant, **one row per trial**, with
two views side by side, both on a **red–blue colormap** (blue = low force, red = high,
matching the paper's dark-blue-0 N → dark-red-peak scale):

- a **3D pyramid**, with the **highest-force area — cells ≥ 70 % of the peak** — rendered
  opaque over a faded full surface (the paper locates its highest-force regions the same way);
- a **2D top view** heatmap of the summed force, with the same **70 %-of-peak contour**
  outlining that high-force region.

Each trial's pyramid is scaled to its own peak, so the 70 % highlight/contour is per trial.

> With the small synthetic sample (a few hundred in-use points per participant) the surface
> is sparse and spiky; on a full recording — many more frames over minutes of work — the same
> code fills the grid into the smooth pyramid shape seen in the paper.


In [ ]:
# ---- Force-pyramid constants (Sawaya et al., Oper Neurosurg 2017) ----------------
FORCE_PYRAMID_VOXEL = 0.5      # mm — xyz rounded to the nearest 0.5 mm grid (paper §Spatial Analysis)
HIGH_FORCE_FRAC     = 0.70     # "highest-force area" = cells >= 70% of the peak pyramid force
FORCE_CMAP          = "RdBu_r" # red-blue: blue = low force, red = high (paper: dark blue 0 N -> dark red peak)

# We record a single overall |force| per timestamp, but several instruments can be in the
# active space at the same frame. Two strategies pick where that one reading lands on the map:
#   "split"   -> share it equally among the instruments in use at that frame (|force|/n_active),
#                so the map's per-frame total equals the sensor reading (no double-counting);
#   "primary" -> attribute the full |force| to PRIMARY_INSTRUMENT's tip only, and build the
#                pyramid from that instrument's in-use frames (closest to a per-hand pyramid).
FORCE_ATTRIBUTION  = "split"       # "split" or "primary"
PRIMARY_INSTRUMENT = "Cavitron"    # tip used when FORCE_ATTRIBUTION == "primary" (e.g. "Cavitron" / "Scissors")


def force_pyramid(P, f, voxel=FORCE_PYRAMID_VOXEL):
    """Force pyramid from registered tip positions P (N,3) and their force f (N,).

    Following the paper: (1) round xyz to the nearest `voxel` mm grid, (2) average the
    force of all samples that share an xyz voxel, (3) sum the per-voxel forces along z
    (depth) to get the total force at each xy cell. Non-finite rows (not-in-use frames,
    which are NaN in `pos_plot`) are dropped, so only in-use samples contribute. Every
    recorded frame is used as-is — no resampling to a fixed rate.
    Returns (GX, GY, Z) grid arrays (Z = summed force per xy, 0 where empty), or None.
    """
    P = np.asarray(P, float); f = np.asarray(f, float)
    keep = np.isfinite(P).all(1) & np.isfinite(f)          # in-use frames carry a position
    P, f = P[keep], f[keep]
    if len(f) == 0:
        return None
    q = np.round(P / voxel).astype(np.int64)               # integer voxel indices
    vox, inv = np.unique(q, axis=0, return_inverse=True)   # (1) unique xyz voxels
    inv = inv.ravel()
    vox_force = np.bincount(inv, weights=f) / np.bincount(inv)          # (2) mean force per xyz voxel
    xy, xyinv = np.unique(vox[:, :2], axis=0, return_inverse=True)
    pyr = np.bincount(xyinv.ravel(), weights=vox_force)                 # (3) sum along z per xy
    ij = xy - xy.min(0)                                    # lay onto a dense grid for plotting
    nx, ny = int(ij[:, 0].max()) + 1, int(ij[:, 1].max()) + 1
    Z = np.zeros((nx, ny))
    Z[ij[:, 0], ij[:, 1]] = pyr
    gx = (xy[:, 0].min() + np.arange(nx)) * voxel
    gy = (xy[:, 1].min() + np.arange(ny)) * voxel
    GX, GY = np.meshgrid(gx, gy, indexing="ij")
    return GX, GY, Z


def pyramid_points(tr, mode=FORCE_ATTRIBUTION, primary=PRIMARY_INSTRUMENT):
    """(registered tip xyz, force) samples for one trial.

    `mode="split"`   — the single overall |force| at each frame is shared equally among the
                       instruments in use there (|force| / n_active), so a co-active frame
                       contributes exactly the sensor reading to the map (no double-counting).
    `mode="primary"` — the full |force| is placed at `primary`'s tip only.
    In-use frames are the non-NaN frames of `pos_plot` (§5). Every recorded frame is used
    as-is; no resampling."""
    f = np.asarray(tr["fmag"], float)
    if mode == "primary":
        P = tr["pos_plot"].get(primary)                   # only the primary tip carries the force
        return (P, f) if P is not None else (None, None)
    # mode == "split": divide the overall force across the instruments active this frame
    insts = [inst for inst in INSTRUMENTS if inst in tr["pos_plot"]]
    if not insts:
        return None, None
    n_active = np.zeros(len(f))
    for inst in insts:
        n_active = n_active + np.isfinite(tr["pos_plot"][inst]).all(1)
    denom = np.where(n_active == 0, 1.0, n_active)         # avoid /0 (those frames are NaN in P anyway)
    fi = f / denom
    Ps = [tr["pos_plot"][inst] for inst in insts]
    Fs = [fi for _ in insts]
    return np.vstack(Ps), np.concatenate(Fs)


def _draw_force_pyramid(ax3d, ax2d, grid, tag):
    """Render one trial's force pyramid onto (ax3d, ax2d); return the top-view mesh or None."""
    if grid is None:
        ax3d.set_axis_off(); ax2d.set_axis_off()
        ax2d.text(0.5, 0.5, "no in-use frames", ha="center", va="center", transform=ax2d.transAxes)
        return None
    GX, GY, Z = grid
    zmax = float(Z.max())
    thr = HIGH_FORCE_FRAC * zmax                            # highest-force area threshold
    Zhi = np.where(Z >= thr, Z, np.nan) if zmax > 0 else np.full_like(Z, np.nan)

    # ---- 3D force pyramid: faded full surface + opaque highest-force area (>= 70% peak) ----
    ax3d.plot_surface(GX, GY, Z, cmap=FORCE_CMAP, vmin=0, vmax=zmax or 1,
                      rstride=1, cstride=1, linewidth=0, antialiased=True, alpha=0.35)
    ax3d.plot_surface(GX, GY, Zhi, cmap=FORCE_CMAP, vmin=0, vmax=zmax or 1,
                      rstride=1, cstride=1, linewidth=0, antialiased=True)
    ax3d.set_title(f"{tag} · 3D pyramid  (≥ {int(HIGH_FORCE_FRAC*100)}% of peak)", fontsize=10)
    ax3d.set_xlabel("x (mm)"); ax3d.set_ylabel("y (mm)"); ax3d.set_zlabel("Σ|force| (N)")
    ax3d.view_init(elev=38, azim=-58)

    # ---- 2D top view: summed-force heatmap + 70%-of-peak contour outlining the high area ----
    pcm = ax2d.pcolormesh(GX, GY, Z, cmap=FORCE_CMAP, vmin=0, vmax=zmax or 1, shading="nearest")
    if zmax > 0 and Z.shape[0] > 1 and Z.shape[1] > 1:
        ax2d.contour(GX, GY, Z, levels=[thr], colors="k", linewidths=1.1)
    ax2d.set_title(f"{tag} · top view (xy)", fontsize=10)
    ax2d.set_xlabel("x (mm)"); ax2d.set_ylabel("y (mm)"); ax2d.set_aspect("equal"); ax2d.grid(False)
    return pcm


_attr_label = ("force split equally" if FORCE_ATTRIBUTION == "split"
               else f"force → {PRIMARY_INSTRUMENT}")

# One figure per participant; one row per trial — 3D pyramid | top view. Each trial's
# pyramid is scaled to its own peak, so the 70% highlight / contour is per trial.
for pid, ptrials in participants.items():
    nrow = len(ptrials)
    fig = plt.figure(figsize=(13, 5.2 * nrow))
    fig.suptitle(f"Force pyramid — participant {pid}   ({_attr_label})", fontweight="bold")
    for r, tr in enumerate(ptrials):
        P, f = pyramid_points(tr)                             # this trial only
        grid = force_pyramid(P, f) if P is not None else None
        ax3d = fig.add_subplot(nrow, 2, 2 * r + 1, projection="3d")
        ax2d = fig.add_subplot(nrow, 2, 2 * r + 2)
        pcm = _draw_force_pyramid(ax3d, ax2d, grid, f"T{tr['trial']}")
        if pcm is not None:
            fig.colorbar(pcm, ax=ax2d, shrink=0.85).set_label("Σ|force| along z (N)")
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()


## Per-trial statistics table

The statistics tables computed by the processing notebook, one row per trial plus a
per-participant aggregate. (The CSV files were written by the processing notebook; here
they are simply displayed.)


In [ ]:
import pandas as pd
from IPython.display import display

per_trial = pd.DataFrame(payload["tables"]["per_trial"])
per_participant = pd.DataFrame(payload["tables"]["per_participant"])

print("Per-trial statistics:")
display(per_trial)
print("Per-participant aggregate (mean over trials):")
display(per_participant)
